# Feature Engineering

Machine Learning models cannot read English words. They need numbers. Here, we use TF-IDF to convert cleaned text into a matrix of numerical features.

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

back to root to call functions from helpers.py file

In [2]:
import os
from pathlib import Path
print(Path.cwd())
os.chdir(Path('..').resolve())
from src.utils.helpers import top_n_grams, save
print(Path.cwd())

/mnt/d/career/digilians/sentiment-analysis-of-amazon-reviews-using-machine-learning/notebooks
/mnt/d/career/digilians/sentiment-analysis-of-amazon-reviews-using-machine-learning


load cleaned text

In [7]:
processed_train = pd.read_csv(r'data/processed/processed_train.csv', dtype=str, quoting=0)
processed_train.head()

,target,title_review,content_review,char_count,word_count,clean_review
0,2,GREAT CAMRA,I HAVE HAD THE DX6340 FOR ABOUT A YEAR.I LOVE ...,586,108,i have had the dx6340 for about a year i love ...
1,1,not so great,I'm using this book in an introductory organic...,570,88,i m using this book in an introductory organic...
2,1,Inaccurate and disappointing,I only read the first few chapters and was bom...,214,40,i only read the first few chapters and was bom...
3,1,Equus 3340,"Feels cheaply made, the battery contacts were ...",193,34,feels cheaply made the battery contacts were ...
4,2,awesome sheets!,I love these sheets! They are sleek & smooth w...,198,38,i love these sheets they are sleek smooth w...


In [8]:
processed_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   target          100 non-null    str  
 1   title_review    100 non-null    str  
 2   content_review  100 non-null    str  
 3   char_count      100 non-null    str  
 4   word_count      100 non-null    str  
 5   clean_review    100 non-null    str  
dtypes: str(6)
memory usage: 4.8 KB


top unigrams and bigrams per class

In [ ]:
feat_eng_train = processed_train.copy()
label_col = 'review_target'

In [ ]:
for cls in feat_eng_train['review_target'].cat.categories if hasattr(feat_eng_train['review_target'],'cat') else feat_eng_train['review_target'].unique():
    subset = feat_eng_train[feat_eng_train['review_target']==cls]['review_cleaned'].astype(str)
    print(f'--- Top unigrams for class {cls} ---')
    print(top_n_grams(subset, ngram_range=(1,1), top_k=15))
    print()
    print(f'--- Top bigrams for class {cls} ---')
    print(top_n_grams(subset, ngram_range=(2,2), top_k=12))
    print('')

--- Top unigrams for class 2 ---
[('book', np.int64(21)), ('use', np.int64(18)), ('good', np.int64(18)), ('great', np.int64(17)), ('just', np.int64(16)), ('like', np.int64(14)), ('read', np.int64(13)), ('time', np.int64(11)), ('easy', np.int64(11)), ('know', np.int64(10)), ('product', np.int64(9)), ('used', np.int64(8)), ('love', np.int64(8)), ('reading', np.int64(8)), ('really', np.int64(7))]

--- Top bigrams for class 2 ---
[('easy use', np.int64(4)), ('long time', np.int64(3)), ('looking forward', np.int64(3)), ('highly recommend', np.int64(3)), ('people know', np.int64(2)), ('works great', np.int64(2)), ('small inconvenience', np.int64(2)), ('good book', np.int64(2)), ('shed stop', np.int64(2)), ('words love', np.int64(2)), ('taping knife', np.int64(2)), ('did know', np.int64(2))]

--- Top unigrams for class 1 ---
[('book', np.int64(25)), ('just', np.int64(20)), ('like', np.int64(17)), ('good', np.int64(13)), ('did', np.int64(12)), ('bought', np.int64(12)), ('don', np.int64(12)), (

In [ ]:
tfidf_train = TfidfVectorizer(ngram_range=(1,2), max_features=20000, stop_words='english')
tfidf_train = tfidf_train.fit_transform(feat_eng_train['review_cleaned'].astype(str))
print('TF-IDF shape:', tfidf_train.shape)


TF-IDF shape: (100, 5202)


Save the tfidf and processed data

In [ ]:
save(base_path='data/processed', dataframe=feat_eng_train, df_name='feat_eng_train.csv', vectorizer=tfidf_train, vectorizer_name='tfidf_train.joblib')